<a href="https://colab.research.google.com/github/Aimagicians/ai-agents-tutorial/blob/main/Outskill_Productivity_AI_Masterclass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI-Assisted Coding Lab

---



A hands-on lab for the **AI-Assisted Coding session** . You will use **one OpenRouter key** to practice everything the live demo covers — on a real repo you can
either build here or clone from GitHub.

> **Why a notebook?** Claude Code itself is a local *agentic* CLI — it reads files, plans, edits, and runs
> your toolchain on your machine, so it cannot run inside Colab. This notebook reproduces the same
> *techniques* through the OpenRouter API so everyone can follow along with just a browser. Each section is
> tagged with the Claude Code feature it mirrors.

### What you will do
1. Connect to OpenRouter
2. **Get a repo** — build the SnipURL demo on disk *or clone any GitHub repo*
3. **Prompt depth ladder** — watch the same task improve from a one-liner to a crafted prompt
4. Core prompt patterns (role, structure, few-shot, self-critique)
5. AI **code review**
6. **Debugging** — reproduce a real bug, fix it, verify it
7. **Documentation** — README + architecture + API reference (on whichever repo you loaded)
8. **Prompt chaining** — a reusable engine + naive-vs-chained, side by side
9. **Plan-first** (spec-driven) mini-demo
10. Exercises

### How each section maps to the live Claude Code demo
| Notebook section | In Claude Code you saw… |
|---|---|
| Get a repo | Running `claude` in a repo + `/init` |
| Depth ladder | Why `CLAUDE.md` context + Plan Mode "think" change results |
| Code review | `/review` |
| Debugging | Plan Mode (investigate before editing) |
| Documentation | Doc generation + the `docx`/`pdf` skills |
| Chaining | Plan Mode + subagents (decomposition) |
| Plan-first | Plan Mode / spec-driven development |


**Running notes**
- Run cells top to bottom; later cells depend on earlier ones.
- Cells that call the model are marked 💬. A full run is ~25 calls — on a shared key, run sections selectively.
- Outputs vary run to run; that is normal for LLMs.

## 1 · Setup

OpenRouter is OpenAI-compatible, so we use the official `openai` client pointed at OpenRouter.
**Best practice:** add your key to **Colab Secrets** (🔑 icon, left sidebar) as `OPENROUTER_KEY`. Otherwise
you will be prompted to paste it (hidden input).

In [ ]:
!pip install -q openai


In [ ]:
from openai import OpenAI
from getpass import getpass
import time

# Pull the key from Colab Secrets first; fall back to a hidden prompt.
OPENROUTER_API_KEY = None
try:
    from google.colab import userdata
    OPENROUTER_API_KEY = userdata.get("OPENROUTER_KEY")
except Exception:
    pass
if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = getpass("Paste your OpenRouter API key (hidden): ")

#@markdown ### Model
#@markdown Pick any chat model id from https://openrouter.ai/models . Use a capable (paid) model — small/free
#@markdown models often miss bugs and break structured output, which defeats the point of the exercises.
MODEL = "anthropic/claude-sonnet-4.5" #@param {type:"string"}

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    default_headers={"X-Title": "SnipURL Masterclass"},
)
print("Client ready. Model:", MODEL)

In [ ]:
def ask(prompt, system="You are a helpful, precise senior software engineer.",
        model=None, max_tokens=1200, temperature=0.3, retries=2):
    """Send one prompt to the model via OpenRouter and return the reply text.

    Low temperature by default for consistent review/debug output. Retries on transient errors.
    """
    last_err = None
    for attempt in range(retries + 1):
        try:
            resp = client.chat.completions.create(
                model=model or MODEL,
                max_tokens=max_tokens,
                temperature=temperature,
                messages=[
                    {"role": "system", "content": system},
                    {"role": "user", "content": prompt},
                ],
            )
            return resp.choices[0].message.content
        except Exception as e:
            last_err = e
            if attempt < retries:
                time.sleep(1.5 * (attempt + 1))
    return ("[API ERROR] " + str(last_err) +
            "\n\nCheck: (1) MODEL is a valid id from https://openrouter.ai/models, "
            "(2) the key has credit, (3) network. Then re-run this cell.")

💬 **Smoke test** — confirm the key + model work before going further.

In [ ]:
print(ask("hey whats up", max_tokens=100))

Hey! Not much, just here and ready to help. What's on your mind? Got any code to write, bugs to squash, or technical problems to solve?


## 2 · Get a repo  · *mirrors: running `claude` in a repo*

You have two options. **Option A** always works (no network, no GitHub account). **Option B** lets you point
the whole lab at a real project — which is exactly the "give me a GitHub repo and generate the docs" use case.

### Option A — Build the SnipURL demo repo on disk (recommended)

A tiny FastAPI URL shortener written to a real folder + `git init`, so the rest of the lab treats it like a
cloned project. It ships with **no docs and a few planted bugs** so the review/debug/doc sections have real work.

In [ ]:
import os, subprocess

# Colab writes to /content; fall back to current dir elsewhere.
BASE = "/content" if os.path.isdir("/content") else "."
DEMO_DIR = os.path.join(BASE, "snipurl")

DEMO_FILES = {
    "models.py": 'from pydantic import BaseModel\n\n\nclass ShortenRequest(BaseModel):\n    url: str\n\n\nclass ShortenResponse(BaseModel):\n    code: str\n    short_url: str\n\n\nclass StatsResponse(BaseModel):\n    code: str\n    long_url: str\n    clicks: int\n',
    "storage.py": '# In-memory store. Swap for a real database (Redis or Postgres) in production.\nclass Storage:\n    def __init__(self):\n        self._urls = {}      # code -> long url\n        self._clicks = {}    # code -> click count\n        self._counter = 0\n\n    def next_id(self):\n        self._counter += 1\n        return self._counter\n\n    def save(self, code, long_url):\n        self._urls[code] = long_url\n        self._clicks[code] = 0\n\n    def get_url(self, code):\n        return self._urls.get(code)\n\n    def increment_clicks(self, code):\n        count = self._clicks.get(code, 0)\n        count + 1  # BUG: result is thrown away; should be self._clicks[code] = count + 1\n        return self._clicks[code]\n\n    def get_clicks(self, code):\n        return self._clicks.get(code, 0)\n',
    "service.py": 'import string\n\nALPHABET = string.ascii_lowercase + string.ascii_uppercase + string.digits  # 62 characters\n\n\ndef encode_base62(num):\n    out = ""\n    while num > 0:\n        out = ALPHABET[num % 62] + out\n        num //= 62\n    return out  # BUG: encode_base62(0) returns empty string instead of a valid one-character code\n\n\ndef record_visit(code, log=[]):  # BUG: mutable default argument is shared across all calls\n    log.append(code)\n    return log\n\n\ndef shorten(storage, long_url):\n    new_id = storage.next_id()\n    code = encode_base62(new_id)\n    storage.save(code, long_url)\n    return code\n',
    "main.py": 'from fastapi import FastAPI\nfrom fastapi.responses import RedirectResponse\n\nfrom models import ShortenRequest, ShortenResponse, StatsResponse\nfrom storage import Storage\nfrom service import shorten, record_visit\n\napp = FastAPI(title="SnipURL", version="0.1.0")\nstore = Storage()\n\n\n@app.post("/shorten", response_model=ShortenResponse)\ndef create_short_url(req: ShortenRequest):\n    code = shorten(store, req.url)  # NOTE: no validation that req.url is a real URL\n    return ShortenResponse(code=code, short_url=f"http://snip.url/{code}")\n\n\n@app.get("/{code}")\ndef redirect_to_url(code: str):\n    long_url = store.get_url(code)\n    store.increment_clicks(code)\n    record_visit(code)\n    return RedirectResponse(long_url)  # BUG: unknown code -> long_url is None; should return 404\n\n\n@app.get("/stats/{code}", response_model=StatsResponse)\ndef get_stats(code: str):\n    return StatsResponse(\n        code=code,\n        long_url=store.get_url(code),\n        clicks=store.get_clicks(code),\n    )\n',
}

subprocess.run(["rm", "-rf", DEMO_DIR])
os.makedirs(DEMO_DIR, exist_ok=True)
for name, src in DEMO_FILES.items():
    with open(os.path.join(DEMO_DIR, name), "w") as f:
        f.write(src)

# Make it a real git repo so it behaves like something you cloned.
subprocess.run(["git", "init", "-q", DEMO_DIR])
subprocess.run(["git", "-C", DEMO_DIR, "add", "."])
subprocess.run(["git", "-C", DEMO_DIR, "-c", "user.email=lab@example.com",
                "-c", "user.name=Lab", "commit", "-q", "-m", "initial"])
print("Built demo repo at", DEMO_DIR)
print("Files:", ", ".join(DEMO_FILES))


Built demo repo at /content/snipurl
Files: models.py, storage.py, service.py, main.py


### Option B — Clone any public GitHub repo (optional)

Paste a URL to point the lab at a real project. Leave it blank to stick with the demo. Big repos are fine —
the loader in the next cell trims content to fit the model's context window and tells you what it kept.

In [ ]:
import subprocess, os

BASE = "/tmp"  # or wherever you want your working directory

REPO_URL = "https://github.com/takshit12/outskill-demo-productivity" #@param {type:"string"}
CLONE_DIR = os.path.join(BASE, "cloned_repo")
if REPO_URL.strip():
    subprocess.run(["rm", "-rf", CLONE_DIR])
    print("Cloning", REPO_URL, "...")
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL.strip(), CLONE_DIR],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print("Cloned to", CLONE_DIR)
    else:
        print("[CLONE FAILED]\n", r.stderr.strip(),
              "\nFalling back to the demo repo.")
        REPO_URL = ""
else:
    print("No URL given - will use the demo repo.")

Cloning https://github.com/takshit12/outskill-demo-productivity ...
Cloned to /tmp/cloned_repo


### Load the repo into context

This walks the chosen repo, skips junk (`.git`, `node_modules`, `__pycache__`, build dirs, binaries),
keeps source/text files, and **enforces a character budget** so the API call stays healthy. It prints a
manifest of exactly what was included.

In [ ]:
import os

SOURCE_EXTS = (".py", ".js", ".jsx", ".ts", ".tsx", ".go", ".rb", ".java", ".rs",
               ".c", ".cpp", ".h", ".cs", ".php", ".swift", ".kt",
               ".md", ".txt", ".toml", ".cfg", ".ini", ".yaml", ".yml", ".json")
SKIP_DIRS = {".git", "node_modules", "__pycache__", ".venv", "venv", "dist", "build",
             ".next", ".idea", ".pytest_cache", "site-packages", ".mypy_cache"}

def load_repo(root, exts=SOURCE_EXTS, max_total_chars=60000, max_file_chars=8000):
    """Walk a repo into {relpath: content} plus a single concatenated REPO_TEXT, within a char budget."""
    files, skipped = {}, []
    for dirpath, dirnames, filenames in os.walk(root):
        dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS]
        for fn in sorted(filenames):
            rel = os.path.relpath(os.path.join(dirpath, fn), root)
            if not fn.lower().endswith(exts):
                skipped.append(rel); continue
            try:
                with open(os.path.join(dirpath, fn), encoding="utf-8") as f:
                    content = f.read()
            except Exception:
                skipped.append(rel); continue
            if len(content) > max_file_chars:
                content = content[:max_file_chars] + "\n... [truncated] ...\n"
            files[rel] = content

    # Enforce the total budget (smaller files first so we keep as many as possible).
    text, used, kept, dropped = [], 0, [], []
    for rel in sorted(files, key=lambda r: len(files[r])):
        block = "# ===== " + rel + " =====\n" + files[rel]
        if used + len(block) > max_total_chars:
            dropped.append(rel); continue
        text.append(block); used += len(block); kept.append(rel)

    repo_text = "\n\n".join(text)
    manifest = {"root": root, "kept": kept, "dropped": dropped,
                "skipped_types": len(skipped), "chars": used, "approx_tokens": used // 4}
    return {r: files[r] for r in kept}, repo_text, manifest

def print_manifest(m):
    print("Repo:", m["root"])
    print("Included", len(m["kept"]), "files (~%d tokens, %d chars):" % (m["approx_tokens"], m["chars"]))
    for r in m["kept"]:
        print("   -", r)
    if m["dropped"]:
        print("Dropped to fit budget:", ", ".join(m["dropped"]))
    print("Skipped (non-source/binary):", m["skipped_types"], "files")


In [ ]:
# Decide which repo to load, then load it.
ACTIVE_DIR = CLONE_DIR if REPO_URL.strip() else DEMO_DIR
REPO, REPO_TEXT, MANIFEST = load_repo(ACTIVE_DIR)
print_manifest(MANIFEST)

Repo: /tmp/cloned_repo
Included 9 files (~1694 tokens, 6776 chars):
   - tests/__init__.py
   - README.md
   - requirements.txt
   - models.py
   - service.py
   - storage.py
   - main.py
   - tests/test_snipurl.py
   - SETUP.md
Skipped (non-source/binary): 1 files


In [ ]:
# Peek at one file (change the name to any path printed above).
PREVIEW = "storage.py" if "storage.py" in REPO else (list(REPO)[0] if REPO else None)
print(REPO.get(PREVIEW, "no files loaded"))

# In-memory store. Swap for a real database (Redis or Postgres) in production.
class Storage:
    def __init__(self):
        self._urls = {}      # code -> long url
        self._clicks = {}    # code -> click count
        self._counter = 0

    def next_id(self):
        self._counter += 1
        return self._counter

    def save(self, code, long_url):
        self._urls[code] = long_url
        self._clicks[code] = 0

    def get_url(self, code):
        return self._urls.get(code)

    def increment_clicks(self, code):
        count = self._clicks.get(code, 0)
        count + 1  # BUG: result is thrown away; should be self._clicks[code] = count + 1
        return self._clicks[code]

    def get_clicks(self, code):
        return self._clicks.get(code, 0)



## 3 · Prompt depth ladder  · *mirrors: why context + "think" change results*

The same request, asked five ways. Run it once and read the outputs as a staircase — each rung adds one
lever (specificity → role → format → an example → self-review). This is the single best way to *feel* why
prompt quality matters, and it is exactly why `CLAUDE.md` (context) and Plan Mode (thinking) improve Claude
Code's answers.

💬 *This cell makes 5 model calls.*

In [ ]:
from IPython.display import Markdown, display

#@markdown Which file should the ladder review?
TARGET_FILE = "storage.py" #@param {type:"string"}
snippet = REPO.get(TARGET_FILE) or REPO[list(REPO)[0]]

L0 = "Review this code:\n\n" + snippet

L1 = ("Review the code below. List ONLY concrete bugs. For each: the function name, what is wrong, "
      "and the one-line fix. Ignore style.\n\n" + snippet)

L2_system = ("You are a senior backend engineer doing a pull-request review. Be specific and do not "
             "invent issues.")
L2 = ("Review the code below. Output a Markdown table with columns: Location, Issue, Severity (High/Med/Low), "
      "Fix. Cover correctness bugs and missing error handling.\n\n" + snippet)

L3 = ("Review the code below and output a Markdown table (Location, Issue, Severity, Fix).\n"
      "Here is the FORMAT and DEPTH I expect, using one example finding:\n"
      "| storage.py: increment_clicks | Computes count+1 but never stores it, so clicks stay 0 | High | "
      "Assign self._clicks[code] = count + 1 |\n"
      "Now produce the full table for ALL issues you find.\n\n" + snippet)

ladder = [
    ("L0  bare one-liner", L0, "You are a helpful assistant.", 0.3),
    ("L1  specific ask", L1, "You are a helpful assistant.", 0.3),
    ("L2  role + format + constraints", L2, L2_system, 0.3),
    ("L3  few-shot (show the format)", L3, L2_system, 0.3),
]

results = {}
for label, prompt, sys, temp in ladder:
    results[label] = ask(prompt, system=sys, temperature=temp, max_tokens=900)

# L4 = self-critique pass built on the L3 answer (the 5th call)
L4 = ("Here is a draft code review:\n\n" + results["L3  few-shot (show the format)"] +
      "\n\nCritique it: remove any false positives, add any real bug it missed, then output the "
      "corrected final table only.")
results["L4  self-critique pass"] = ask(L4, system=L2_system, max_tokens=900)

for label in ["L0  bare one-liner", "L1  specific ask", "L2  role + format + constraints",
              "L3  few-shot (show the format)", "L4  self-critique pass"]:
    display(Markdown("### " + label))
    display(Markdown(results[label]))


### L0  bare one-liner

# Code Review

## Critical Bug Found 🐛

**Line 18** in `increment_clicks()`:
```python
count + 1  # BUG: result is thrown away
```

**Should be:**
```python
self._clicks[code] = count + 1
```

The current code calculates `count + 1` but doesn't store it, so clicks are never actually incremented.

---

## Additional Issues & Recommendations

### 1. **Thread Safety** ⚠️
This implementation is **not thread-safe**. Multiple concurrent requests could cause:
- Race conditions in `increment_clicks()`
- Duplicate IDs from `next_id()`

**Solutions:**
- Use `threading.Lock()` for synchronization
- Or switch to thread-safe alternatives (Redis, database with transactions)

### 2. **Inconsistent State Handling**
`increment_clicks()` uses `.get(code, 0)` but then accesses `self._clicks[code]` directly, which could raise `KeyError` if the code doesn't exist.

**Fix:**
```python
def increment_clicks(self, code):
    self._clicks[code] = self._clicks.get(code, 0) + 1
    return self._clicks[code]
```

### 3. **Missing Error Handling**
No validation for:
- Empty/None URLs
- Duplicate codes
- Invalid code formats

### 4. **Memory Leak Risk**
No mechanism to clean up old/expired URLs. Consider adding TTL or max entries limit.

### 5. **Code Simplification**
You could use `collections.defaultdict(int)` for `_clicks` to eliminate `.get(code, 0)` patterns.

---

## Corrected Version

```python
from collections import defaultdict
from threading import Lock

class Storage:
    def __init__(self):
        self._urls = {}
        self._clicks = defaultdict(int)
        self._counter = 0
        self._lock = Lock()

    def next_id(self):
        with self._lock:
            self._counter += 1
            return self._counter

    def save(self, code, long_url):
        with self._lock:
            self._urls[code] = long_url
            self._clicks[code] = 0

    def get_url(self, code):
        return self._urls.get(code)

    def increment_clicks(self, code):
        with self._lock:
            self._clicks[code] += 1
            return self._clicks[code]

    def get_clicks(self, code):
        return self._clicks[code]  # defaultdict returns 0 for missing keys
```

### L1  specific ask

# Concrete Bugs

1. **Function:** `increment_clicks`
   **Bug:** `count + 1` computes the result but doesn't store it
   **Fix:** `self._clicks[code] = count + 1`

### L2  role + format + constraints

# Pull Request Review

| Location | Issue | Severity | Fix |
|----------|-------|----------|-----|
| `increment_clicks()`, line 19 | Result of `count + 1` is not assigned. Click counts will never increment. | High | Change `count + 1` to `self._clicks[code] = count + 1` |
| `increment_clicks()`, line 20 | Returns stale value (0) instead of incremented value due to bug on line 19 | High | After fixing line 19, change to `return count + 1` or `return self._clicks[code]` |
| `next_id()` | Race condition: not thread-safe. Multiple threads can get the same ID. | High | Use `threading.Lock()` around counter increment, or use `itertools.count()` with thread-safe operations |
| `save()` | No validation that `code` doesn't already exist. Silently overwrites existing URLs. | Medium | Check if code exists and raise exception, or return boolean success indicator |
| `save()` | No validation of `long_url` parameter (could be None, empty string, or invalid URL) | Medium | Add validation to ensure `long_url` is a non-empty string |
| `increment_clicks()` | No check if `code` exists before incrementing. Will create entry with count 1 for non-existent codes. | Medium | Check if `code` exists in `self._urls` first, raise KeyError or return None if not found |
| `get_clicks()` | Returns 0 for non-existent codes, indistinguishable from codes with zero clicks | Low | Consider returning `None` for non-existent codes or raising KeyError for consistency |
| `save()` | Race condition: not thread-safe. Multiple threads could corrupt `_urls` and `_clicks` dictionaries. | Medium | Use `threading.Lock()` around dictionary operations if multi-threaded access is expected |

**Summary**: The critical bug is the missing assignment in `increment_clicks()`. Thread-safety issues are also significant if this will handle concurrent requests (typical for web services). Input validation would prevent data integrity issues.

### L3  few-shot (show the format)

| Location | Issue | Severity | Fix |
|----------|-------|----------|-----|
| storage.py: increment_clicks | Computes count+1 but never stores it, so clicks stay 0 | High | Assign self._clicks[code] = count + 1 |

### L4  self-critique pass

I need to see the actual code to properly critique this review. However, based on the review comment alone, I can analyze its validity:

The review identifies a classic bug pattern where a value is computed but not stored back. This is a legitimate issue IF:
1. The code actually computes `count + 1` without assignment
2. There's no return value being used elsewhere
3. The method is supposed to mutate state (increment clicks in place)

Without seeing the actual code, I cannot:
- Confirm this is a true positive
- Identify any missed bugs
- Verify the suggested fix is correct

**I cannot provide a corrected review table without seeing the actual code from storage.py.**

Please provide the code being reviewed, and I'll give you an accurate critique with a corrected table showing only real issues.

**What changed as you climbed?** Usually: L0 rambles, L1 gets concrete, L2 becomes scannable and
prioritized, L3 matches your exact format, and L4 catches a miss or drops a false positive. Same model — the
only variable was how you asked.

## 4 · Core prompt patterns

Four reusable levers. Each cell shows the lever in action; the contrast is the lesson.

### 4.1 Role priming 💬 — *who* you ask it to be shifts *what* it notices

In [ ]:
main_file = REPO.get("main.py") or REPO[list(REPO)[0]]
neutral  = ask("Tell me about this file:\n\n" + main_file, max_tokens=500)
security = ask("Tell me about this file:\n\n" + main_file, max_tokens=500,
               system="You are a paranoid application-security reviewer. You care most about missing input "
                      "validation, error handling, and unsafe responses.")
print("--- NEUTRAL ---\n", neutral)
print("\n\n--- SECURITY LENS ---\n", security)

--- NEUTRAL ---
 # File Analysis: URL Shortener Service

This is a **FastAPI-based URL shortening service** (like bit.ly or TinyURL) called "SnipURL". Here's a breakdown:

## Architecture

**Main Components:**
- `FastAPI` web framework
- `Storage` class for data persistence
- Service functions (`shorten`, `record_visit`)
- Three endpoints with Pydantic models for request/response validation

## Endpoints

### 1. `POST /shorten` - Create Short URL
- **Input**: `ShortenRequest` with a long URL
- **Output**: `ShortenResponse` with generated code and short URL
- **Flow**: Generates a short code and returns formatted short URL

### 2. `GET /{code}` - Redirect to Original URL
- **Input**: Short code from URL path
- **Output**: HTTP redirect to original URL
- **Side effects**: Increments click counter and records visit
- **Flow**: Looks up original URL → tracks analytics → redirects user

### 3. `GET /stats/{code}` - Get URL Statistics
- **Input**: Short code from URL path
- **Output**: `Stat

### 4.2 Structured output 💬 — specify the shape when you will reuse the result

In [ ]:
from IPython.display import Markdown, display
table = ask("From the code below, produce a Markdown table of every HTTP endpoint with columns: "
            "Method, Path, Request body, Response, Notes. Output ONLY the table.\n\n" + REPO_TEXT,
            max_tokens=900)
display(Markdown(table))

| Method | Path | Request body | Response | Notes |
|--------|------|--------------|----------|-------|
| POST | `/shorten` | `{"url": "string"}` | `{"code": "string", "short_url": "string"}` | Creates a shortened URL; no validation that url is a real URL |
| GET | `/{code}` | None | RedirectResponse | Redirects to long URL and increments click counter; BUG: returns None instead of 404 for unknown codes |
| GET | `/stats/{code}` | None | `{"code": "string", "long_url": "string", "clicks": int}` | Returns statistics for a shortened URL |

### 4.3 Few-shot 💬 — one good example beats a paragraph of description

In [ ]:
fewshot = ask(
    "Write docstrings for the functions below. Match THIS style exactly:\n"
    "\"\"\"Return X for the given Y. Raises ValueError if Y is invalid.\"\"\"\n"
    "One line, imperative mood, note what it raises. Output the functions with docstrings added.\n\n"
    + (REPO.get("service.py") or REPO[list(REPO)[0]]),
    max_tokens=900)
print(fewshot)

```python
import string

ALPHABET = string.ascii_lowercase + string.ascii_uppercase + string.digits  # 62 characters


def encode_base62(num):
    """Return base62 encoded string for the given number. Raises ValueError if num is negative."""
    out = ""
    while num > 0:
        out = ALPHABET[num % 62] + out
        num //= 62
    return out  # BUG: encode_base62(0) returns "" instead of a valid one-character code


def record_visit(code, log=[]):  # BUG: mutable default argument is shared across all calls
    """Return log list with the given code appended. Raises AttributeError if log is not a list."""
    log.append(code)
    return log


def shorten(storage, long_url):
    """Return shortened code for the given long_url. Raises AttributeError if storage lacks required methods."""
    new_id = storage.next_id()
    code = encode_base62(new_id)
    storage.save(code, long_url)
    return code
```


### 4.4 Self-critique 💬 — ask it to review and improve its own answer

In [ ]:
draft = ask("Write a one-paragraph description of what this project does:\n\n" + REPO_TEXT, max_tokens=300)
improved = ask("Here is a draft description:\n\n" + draft +
               "\n\nCritique it for accuracy and vagueness, then output an improved final version only in ALL CAPS.",
               max_tokens=300)
print("--- DRAFT ---\n", draft, "\n\n--- IMPROVED ---\n", improved)

--- DRAFT ---
 **SnipURL** is a minimal URL shortening service built with FastAPI that converts long URLs into short, shareable codes using base62 encoding. Users can POST a long URL to `/shorten` to receive a compact code (e.g., "b" for ID 1, "c" for ID 2), visit `/{code}` to be redirected to the original URL while incrementing a click counter, and query `/stats/{code}` to retrieve analytics including the original URL and total clicks. The project uses an in-memory storage backend (suitable for demos, with notes to swap in Redis or Postgres for production), includes Pydantic models for request/response validation, and ships with a test suite that intentionally documents three planted bugs—a non-incrementing click counter, a base62 encoder that fails on zero, and a mutable default argument—making it an ideal teaching tool for debugging, code review, and documentation generation exercises. 

--- IMPROVED ---
 # Critique

**Accuracy issues:**
1. "b" for ID 1 is incorrect - base62 encodin

## 5 · AI code review  · *mirrors: `/review`* 💬

A senior-reviewer persona over the whole repo, asked for specific, actionable findings. On the demo repo it
should surface the planted bugs (click counter, mutable default arg, missing 404, base62 zero edge, no URL
validation).

In [ ]:
review = ask(
    "Review the repository below. Find correctness bugs, missing error handling, and security issues. "
    "For each finding give: file, the problem, and a concrete fix. Be specific; do not invent issues.\n\n"
    + REPO_TEXT,
    system="You are a senior backend engineer doing a thorough pull-request review.",
    max_tokens=1600)
print(review)

# Pull Request Review: SnipURL

## Critical Bugs

### 1. **storage.py** - Click counter never increments
**File:** `storage.py`, line 21  
**Problem:** The `increment_clicks` method computes `count + 1` but discards the result instead of storing it.
```python
def increment_clicks(self, code):
    count = self._clicks.get(code, 0)
    count + 1  # BUG: result is thrown away
    return self._clicks[code]
```
**Fix:**
```python
def increment_clicks(self, code):
    count = self._clicks.get(code, 0)
    self._clicks[code] = count + 1
    return self._clicks[code]
```

### 2. **service.py** - encode_base62 returns empty string for zero
**File:** `service.py`, line 6-11  
**Problem:** When `num=0`, the while loop never executes, returning an empty string instead of a valid code.
```python
def encode_base62(num):
    out = ""
    while num > 0:  # never enters when num=0
        out = ALPHABET[num % 62] + out
        num //= 62
    return out
```
**Fix:**
```python
def encode_base62(num):
   

## 6 · Debugging  · *mirrors: Plan Mode (investigate before editing)*

Good debugging starts from a **reproduction**, not a guess. The next cell actually runs the demo repo's
`Storage` class so you can *see* the click-counter bug. *(This reproduction is specific to the demo repo.)*

In [ ]:
# Run the real Storage class from the demo files and watch the bug.
import sys, os

ns = {}

if "DEMO_FILES" in dir():
    # Option A: use the in-memory dict
    exec(DEMO_FILES["storage.py"], ns)
else:
    # Option B: read from the cloned repo on disk
    storage_path = os.path.join(CLONE_DIR, "storage.py")
    with open(storage_path) as f:
        exec(f.read(), ns)

s = ns["Storage"]()
s.save("abc", "https://example.com")
s.increment_clicks("abc"); s.increment_clicks("abc")
print("Clicks after 2 visits (should be 2):", s.get_clicks("abc"))  # prints 0 -> the bug

Clicks after 2 visits (should be 2): 0


💬 Now hand the model the reproduction and make it reason before fixing.

In [ ]:
fix = ask(
    "This class should count clicks, but get_clicks() returns 0 after increment_clicks() is called. "
    "Reproduction: save('abc', url); increment_clicks('abc') x2; get_clicks('abc') -> 0. "
    "Think step by step about the cause, then give the corrected method only.\n\n"
    + DEMO_FILES["storage.py"],
    system="You are debugging Python. Reason step by step before giving the fix.",
    max_tokens=700)
print(fix)

Let me trace through the execution step by step:

1. `save('abc', url)` → sets `self._clicks['abc'] = 0`
2. `increment_clicks('abc')` (first call):
   - `count = self._clicks.get('abc', 0)` → `count = 0`
   - `count + 1` → evaluates to `1` but **the result is not stored anywhere**
   - `return self._clicks['abc']` → returns `0` (unchanged)
3. `increment_clicks('abc')` (second call):
   - Same issue, `self._clicks['abc']` is still `0`
4. `get_clicks('abc')` → returns `0`

**The bug:** Line `count + 1` computes the new value but doesn't assign it back to the dictionary.

**Corrected method:**

```python
def increment_clicks(self, code):
    count = self._clicks.get(code, 0)
    self._clicks[code] = count + 1
    return self._clicks[code]
```

Alternatively, a more concise version:

```python
def increment_clicks(self, code):
    self._clicks[code] = self._clicks.get(code, 0) + 1
    return self._clicks[code]
```


**Verify the fix yourself.** Edit the `increment_clicks` line in *Option A* (Section 2) to
`self._clicks[code] = count + 1`, re-run that cell and the reproduction cell — you should now see `2`.
Watching the test go green is the entire point of reproduce-first debugging.

## 7 · Documentation  · *mirrors: doc generation + the `docx`/`pdf` skills*

The headline use case: point the model at a repo and generate the docs nobody wrote. These cells run on
**whatever repo you loaded** in Section 2 — demo or your clone.

### 7.1 README 💬

In [ ]:
from IPython.display import Markdown, display
readme = ask(
    "Write a clear skill.md for the project below. Include: a one-line summary, key features, how to "
    "install and run it, an example for each entry point or endpoint, and the project structure. "
    "Output Markdown only.\n\n" + REPO_TEXT,
    max_tokens=1600)
display(Markdown(readme))

# SnipURL

A tiny URL shortener built with FastAPI that generates short codes for long URLs and tracks click statistics.

## Key Features

- **URL Shortening** – Convert long URLs into short, base62-encoded codes
- **Redirection** – Automatically redirect short codes to original URLs
- **Click Tracking** – Monitor how many times each short URL has been accessed
- **In-Memory Storage** – Simple storage implementation (swap for Redis/Postgres in production)
- **REST API** – Clean FastAPI endpoints with automatic OpenAPI documentation

## Installation

```bash
# Clone the repository
git clone https://github.com/<you>/snipurl.git
cd snipurl

# Create and activate virtual environment (optional)
python -m venv .venv
source .venv/bin/activate  # On Windows: .venv\Scripts\activate

# Install dependencies
pip install -r requirements.txt
```

## Running the Application

```bash
# Start the development server
uvicorn main:app --reload

# The API will be available at http://127.0.0.1:8000
# Interactive API docs at http://127.0.0.1:8000/docs
```

## Running Tests

```bash
pytest -q
```

## API Examples

### 1. Shorten a URL

**Endpoint:** `POST /shorten`

```bash
curl -X POST http://127.0.0.1:8000/shorten \
  -H "Content-Type: application/json" \
  -d '{"url": "https://example.com/very/long/url"}'
```

**Response:**
```json
{
  "code": "b",
  "short_url": "http://snip.url/b"
}
```

### 2. Redirect to Original URL

**Endpoint:** `GET /{code}`

```bash
curl -L http://127.0.0.1:8000/b
```

This will redirect (HTTP 307) to the original long URL.

### 3. Get URL Statistics

**Endpoint:** `GET /stats/{code}`

```bash
curl http://127.0.0.1:8000/stats/b
```

**Response:**
```json
{
  "code": "b",
  "long_url": "https://example.com/very/long/url",
  "clicks": 2
}
```

## Project Structure

```
snipurl/
├── main.py                   # FastAPI application with route handlers
├── service.py                # Business logic: base62 encoding, URL shortening
├── storage.py                # In-memory storage implementation
├── models.py                 # Pydantic models for request/response validation
├── requirements.txt          # Python dependencies
├── README.md                 # This file
├── SETUP.md                  # Setup instructions for masterclass demos
└── tests/
    ├── __init__.py
    └── test_snipurl.py      # Unit tests (1 passing, 2 xfail for planted bugs)
```

## Known Issues

This project contains intentional bugs for educational purposes:

- Click counter does not increment properly
- Base62 encoding fails for zero values
- Visit logging uses mutable default arguments

See `tests/test_snipurl.py` for documented test cases.

## Production Considerations

- Replace in-memory storage with Redis or PostgreSQL
- Add URL validation for the `/shorten` endpoint
- Implement proper error handling (404 for unknown codes)
- Add rate limiting and authentication
- Use environment variables for configuration

### 7.2 Architecture diagram (Mermaid) 💬

In [ ]:
mermaid = ask(
    "Generate a Mermaid \"flowchart LR\" showing the main components of this project and how a typical "
    "request or call flows through them. Output ONLY a fenced mermaid code block.\n\n" + REPO_TEXT,
    max_tokens=700)
print(mermaid)
print("\nPaste this into https://mermaid.live or a GitHub .md file to render it.")

### 7.3 API reference (structured output) 💬

In [ ]:
api_ref = ask(
    "Produce an API reference as Markdown. For each endpoint or public function include: name/path, "
    "parameters, inputs, outputs, and error/status behavior. Use the code below as the source of truth.\n\n"
    + REPO_TEXT,
    max_tokens=1600)
display(Markdown(api_ref))

## 8 · Prompt chaining  · *mirrors: Plan Mode + subagents (decomposition)*

One mega-prompt does everything shallowly and in the wrong order. **Chaining** breaks the task into ordered
steps where each output feeds the next. Below is a tiny reusable engine, then a side-by-side comparison.

In [ ]:
def run_chain(steps, verbose=True):
    """Run an ordered list of steps. Each step is a dict:
        name:   label for the output
        build:  function(context_dict) -> prompt string  (can reference earlier outputs)
        system: optional system prompt
        max_tokens: optional int
    Returns a dict {name: output}.
    """
    context = {}
    for i, step in enumerate(steps, 1):
        prompt = step["build"](context)
        out = ask(prompt, system=step.get("system", "You are a precise senior software engineer."),
                  max_tokens=step.get("max_tokens", 1200))
        context[step["name"]] = out
        if verbose:
            print("\n" + "=" * 70)
            print("STEP %d - %s" % (i, step["name"]))
            print("=" * 70)
            print(out[:1500] + (" ..." if len(out) > 1500 else ""))
    return context


### 8.1 Naive — one prompt for everything 💬
Tends to skim each task, miss bugs, and (worst of all) document the *buggy* behavior as if it were correct.

In [ ]:
naive = ask(
    "Review this repo, fix all the bugs, write docstrings, and generate a README - all in one response.\n\n"
    + REPO_TEXT,
    max_tokens=6000)
print(naive[:2500], "...")

# Complete Review, Bug Fixes, Documentation & README

## 🐛 Bugs Found & Fixed

### 1. **storage.py** - `increment_clicks` doesn't save the result
```python
# BEFORE (line 23-25):
def increment_clicks(self, code):
    count = self._clicks.get(code, 0)
    count + 1  # BUG: result is thrown away
    return self._clicks[code]

# AFTER:
def increment_clicks(self, code):
    """Increment the click counter for a given short code.
    
    Args:
        code: The short URL code
        
    Returns:
        The new click count
    """
    count = self._clicks.get(code, 0)
    self._clicks[code] = count + 1
    return self._clicks[code]
```

### 2. **service.py** - `encode_base62(0)` returns empty string
```python
# BEFORE (line 6-10):
def encode_base62(num):
    out = ""
    while num > 0:
        out = ALPHABET[num % 62] + out
        num //= 62
    return out  # BUG: returns "" for 0

# AFTER:
def encode_base62(num):
    """Encode a non-negative integer to base62 string.
    
    Args:
    

### 8.2 Chained — Explore → Find bugs → Fix → Test → Document 💬💬💬💬💬
Each step builds on a *correct* previous step. Note how step 3 receives step 2's bug list, and step 4/5
operate on the *fixed* code — so the docs describe correct behavior. *(5 model calls.)*

In [ ]:
steps = [
    {"name": "explore",
     "build": lambda c: "Summarize what each file in this repo does, 1-2 lines each.\n\n" + REPO_TEXT,
     "max_tokens": 2000},
    {"name": "bugs",
     "build": lambda c: "List every correctness bug in this repo as a numbered list. One line each: "
                        "file + problem.\n\n" + REPO_TEXT,
     "max_tokens": 2000},
    {"name": "fixes",
     "build": lambda c: "Here is a confirmed bug list:\n\n" + c["bugs"] +
                        "\n\nOutput corrected code for each affected file. Code only, grouped by filename."
                        "\n\n" + REPO_TEXT,
     "max_tokens": 2000},
    {"name": "tests",
     "build": lambda c: "Write pytest tests that would PASS on this corrected code and would have FAILED "
                        "on the original bugs:\n\n" + c["fixes"],
     "max_tokens": 2000},
    {"name": "readme",
     "build": lambda c: "Using this corrected code, write a short, accurate README "
                        "(summary, endpoints, run steps):\n\n" + c["fixes"],
     "max_tokens": 2000},
]
chain_out = run_chain(steps)
print("\n\nChain complete. Available outputs:", list(chain_out.keys()))


STEP 1 - explore
# File Summary

## Core Application Files

**main.py** - FastAPI app with 3 endpoints: POST /shorten (create short URL), GET /{code} (redirect), GET /stats/{code} (view click stats); contains bug where unknown codes don't return 404.

**service.py** - Business logic for base62 encoding and URL shortening; contains 2 bugs: encode_base62(0) returns empty string, and record_visit() uses mutable default argument.

**storage.py** - In-memory storage class managing URLs, click counts, and ID generation; contains bug where increment_clicks() calculates but doesn't save the incremented value.

**models.py** - Pydantic models defining request/response schemas: ShortenRequest, ShortenResponse, and StatsResponse.

## Configuration & Documentation

**requirements.txt** - Python dependencies: FastAPI, Uvicorn, Pydantic, httpx, and pytest.

**README.md** - Minimal one-line description stating it's a URL shortener with TODO for proper docs (intentionally bare for demo purposes).

**

**Takeaway:** chaining is not "more prompts" — it is *ordering the work* so each step stands on solid
ground. Find bugs before fixing; fix before documenting; document correct code. In Claude Code, **Plan Mode**
is the plan step and **subagents** are the decomposition — the same idea, automated.

## 9 · Plan-first / spec-driven  · *mirrors: Plan Mode & spec-driven development*

The cheapest place to fix a misunderstanding is *before* any code exists. Ask for a short spec, read it,
*then* implement. A wrong plan is a one-line correction; wrong code is a debugging session.

In [ ]:
# Step 1: spec only (no code yet) - this is the "Plan Mode" gate.
spec = ask(
    "We want to add validation so /shorten rejects anything that is not a valid http/https URL. "
    "Produce a SHORT spec ONLY (no code): goal, files to change and why, the new behavior, edge cases, "
    "and a 3-line test plan.\n\n" + REPO_TEXT,
    max_tokens=700)
print("--- SPEC ---\n", spec)

In [ ]:
# Step 2: implement exactly the approved spec.
impl = ask(
    "Implement exactly this approved spec. Output only the changed code, grouped by filename, with a "
    "one-line note above each change.\n\nSPEC:\n" + spec + "\n\nCURRENT CODE:\n" + REPO_TEXT,
    max_tokens=1400)
print("--- IMPLEMENTATION ---\n", impl)

## 10 · Your turn

1. **Write your own reviewer.** Fill in the system prompt below to focus *only* on performance, then run it.
2. **Add a chain step.** In Section 8.2, add a step after `fixes` that asks for a CHANGELOG entry.
3. **Force a schema.** Change the API-reference cell (7.3) to output valid **OpenAPI JSON** instead of Markdown.
4. **Clone something real.** Put a GitHub URL in Section 2 Option B, reload, and regenerate the README + diagram.
5. **Swap the model.** Change `MODEL` to another id from [openrouter.ai/models](https://openrouter.ai/models) and re-run the depth ladder — compare. (Weaker models often miss bugs or break the table: a lesson in itself.)

In [ ]:
# EXERCISE 1 - your own reviewer persona.
my_system = ""   # <- describe a performance-focused reviewer here
print(ask("Review this file for performance issues:\n\n" + (REPO.get("service.py") or REPO[list(REPO)[0]]),
          system=my_system or "You are a code reviewer.", max_tokens=700))

## Recap

- **Context + constraints** decide answer quality far more than the model does — the depth ladder proved it.
- **Reproduce** bugs before fixing; **verify** by re-running.
- **Chain** prompts so each step builds on a correct one; **plan before** you generate.
- Everything here maps to a Claude Code feature: `/review`, Plan Mode, subagents, doc skills, `CLAUDE.md`.

**Go deeper:** Claude Code docs (Plan Mode, subagents, skills) and the prompt-engineering guide at
`docs.claude.com`.